# Phase 03: Multimodal Dataset Labeling for Four-Class Workload Proxy Classification

This notebook constructs final run-level modeling datasets for task-difficulty-induced workload proxy classification. The target labels are difficulty levels 1, 2, 3, and 4 encoded as targets 0, 1, 2, and 3. These are workload proxy labels induced by task difficulty, not direct psychological stress or questionnaire-based workload labels. No ML model, HDC model, global scaling, or global imputation is performed in this phase.

In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
PHASE_DIR = PROJECT_ROOT / 'experiments' / 'phase_03_multimodal_dataset_labeling'
sys.path.insert(0, str(PHASE_DIR / 'scripts'))
from phase03_dataset import run_phase03

artifacts = run_phase03(PROJECT_ROOT)

print(f"Loaded Phase 02 feature table shape: {artifacts['phase02_shape']}")
display(artifacts['phase02_head'])
print('Identifier column check:')
display(artifacts['identifier_check'])

Loaded Phase 02 feature table shape: (487, 1252)


,subject_id,session_id,run_id,difficulty_level,run_key,ecg_lslshimmerecg_duration,ecg_lslshimmerecg_ecg_projection_la_ra_mv_iqr,ecg_lslshimmerecg_ecg_projection_la_ra_mv_kurtosis,ecg_lslshimmerecg_ecg_projection_la_ra_mv_max,ecg_lslshimmerecg_ecg_projection_la_ra_mv_mean,...,xplane_lslxp11xpcac_aircraft_yaw_deg_missing_ratio,xplane_lslxp11xpcac_aircraft_yaw_deg_num_missing,xplane_lslxp11xpcac_aircraft_yaw_deg_range,xplane_lslxp11xpcac_aircraft_yaw_deg_rms,xplane_lslxp11xpcac_aircraft_yaw_deg_skew,xplane_lslxp11xpcac_aircraft_yaw_deg_slope,xplane_lslxp11xpcac_aircraft_yaw_deg_std,xplane_lslxp11xpcac_duration,xplane_lslxp11xpcac_sample_count,xplane_lslxp11xpcac_sampling_rate_estimate
0,sub-cp003,ses-20210206,run-001,level-000,sub-cp003_ses-20210206_level-000_run-001,324.413710,0.863729,-0.649131,-0.316374,-1.676774,...,0.0,0.0,0.000000,0.014609,NaN,0.000000,1.735157e-18,323.440259,2003.0,6.189706
1,sub-cp003,ses-20210206,run-002,level-000,sub-cp003_ses-20210206_level-000_run-002,328.206548,0.437634,0.001936,-3.152869,-4.275732,...,0.0,0.0,0.000000,0.000000,NaN,NaN,0.000000e+00,NaN,1.0,NaN
2,sub-cp003,ses-20210206,run-001,level-01B,sub-cp003_ses-20210206_level-01B_run-001,535.260675,0.535623,16.777445,5.340462,-1.481550,...,0.0,0.0,3.811121,0.437767,0.719148,-0.001203,4.336922e-01,534.491448,2407.0,4.501475
3,sub-cp003,ses-20210206,run-007,level-01B,sub-cp003_ses-20210206_level-01B_run-007,472.294350,0.281178,-0.415114,-3.523767,-4.357186,...,0.0,0.0,6.937821,0.556384,-0.077715,-0.005958,5.554413e-01,471.972053,2126.0,4.502385
4,sub-cp003,ses-20210206,run-012,level-01B,sub-cp003_ses-20210206_level-01B_run-012,472.847496,0.405901,-0.325430,-2.776779,-4.177590,...,0.0,0.0,3.484196,0.358065,0.559158,0.000458,3.580341e-01,471.853486,2125.0,4.501397


Identifier column check:


,column,present
0,subject_id,True
1,session_id,True
2,run_id,True
3,difficulty_level,True
4,run_key,True


## Difficulty Labels and Target Encoding

The notebook parses difficulty labels, keeps only levels 1-4, removes duplicate `run_key` rows if present, and creates `target = difficulty_level - 1`.

In [2]:
print('Difficulty level distribution before filtering, as loaded from Phase 02:')
display(artifacts['raw_difficulty_distribution'])
print('Difficulty level distribution before filtering, after numeric parsing:')
display(artifacts['numeric_difficulty_distribution_before'])
print('Difficulty level distribution after keeping levels 1, 2, 3, and 4:')
display(artifacts['difficulty_distribution_after'])
print('Target mapping:')
display(artifacts['target_mapping_table'])

Difficulty level distribution before filtering, as loaded from Phase 02:


,raw_difficulty_level,sample_count
0,level-02B,106
1,level-04B,105
2,level-01B,104
3,level-03B,104
4,level-000,68


Difficulty level distribution before filtering, after numeric parsing:


,difficulty_level,sample_count
0,0,68
1,1,104
2,2,106
3,3,104
4,4,105


Difficulty level distribution after keeping levels 1, 2, 3, and 4:


,difficulty_level,sample_count
0,1,104
1,2,106
2,3,104
3,4,105


Target mapping:


,difficulty_level,target
0,1,0
1,2,1
2,3,2
3,4,3


## Dataset Construction and Feature Audits

Identifier and label columns are preserved for interpretation but excluded from input feature lists. Performance metrics are separated from the main dataset to reduce shortcut-learning risk.

In [3]:
print('Class distribution table:')
display(artifacts['class_distribution'])
print('Subject distribution table:')
display(artifacts['subject_distribution'])
print('Missing value summary, top 20 by missing ratio:')
display(artifacts['missing_value_summary'].head(20))
print('Missing value by feature group:')
display(artifacts['missing_by_group'])
print('Feature group summary without performance:')
display(artifacts['feature_count_by_group_without'])
print('Feature group summary with performance:')
display(artifacts['feature_count_by_group_with'])
print(f"Number of features without performance metrics: {artifacts['feature_count_without_performance']}")
print(f"Number of features with performance metrics: {artifacts['feature_count_with_performance']}")
print('Removed high-missing features:')
display(artifacts['removed_features_high_missing'])
print('Dropped non-numeric columns:')
display(artifacts['dropped_non_numeric_columns'])

Class distribution table:


,difficulty_level,target,sample_count,proportion
0,1,0,104,0.248210
1,2,1,106,0.252983
2,3,2,104,0.248210
3,4,3,105,0.250597


Subject distribution table:


,subject_id,sample_count,target_0_count,target_1_count,target_2_count,target_3_count
0,sub-cp003,12,3,3,3,3
1,sub-cp004,12,3,3,3,3
2,sub-cp005,12,3,3,3,3
3,sub-cp006,12,3,3,3,3
4,sub-cp008,12,3,3,3,3
5,sub-cp009,12,3,3,3,3
6,sub-cp011,12,3,3,3,3
7,sub-cp012,12,3,3,3,3
8,sub-cp013,12,3,3,3,3
9,sub-cp014,12,3,3,3,3


Missing value summary, top 20 by missing ratio:


...

,feature_group,feature_count
4,control_input_features,0
1,eye_tracking_features,416
3,flight_parameter_features,326
2,head_movement_features,159
5,performance_features,59
0,physiological_features,233
6,unknown_features,42


Number of features without performance metrics: 1176
Number of features with performance metrics: 1235
Removed high-missing features:


,feature,missing_ratio,feature_group
0,eye_lslhtcviveeye_convergence_distance_validit...,1.0,eye_tracking_features
1,eye_lslhtcviveeye_validity_r_skew,1.0,eye_tracking_features
2,eye_lslhtcviveeye_validity_l_skew,1.0,eye_tracking_features
3,eye_lslhtcviveeye_convergence_distance_mm_skew,1.0,eye_tracking_features
4,eye_lslhtcviveeye_convergence_distance_mm_kurt...,1.0,eye_tracking_features
5,eye_lslhtcviveeye_validity_r_kurtosis,1.0,eye_tracking_features
6,eye_lslhtcviveeye_pupil_position_l_x_skew,1.0,eye_tracking_features
7,xplane_lslxp11xpcac_aircraft_landing_gear_kurt...,1.0,flight_parameter_features
8,eye_lslhtcviveeye_convergence_distance_validit...,1.0,eye_tracking_features
9,eye_lslhtcviveeye_validity_l_kurtosis,1.0,eye_tracking_features


Dropped non-numeric columns:


,column,reason


## Leakage Check, GroupKFold Readiness, and Final Comparison

The final outputs include two dataset versions, feature-column files, feature-group files, leakage and construction reports, summary tables, figures, logs, README updates, and experiment status updates.

In [4]:
print('Leakage check conclusion:')
display(Markdown(artifacts['leakage_check_report']['conclusion']))
print('GroupKFold readiness conclusion:')
display(Markdown(artifacts['groupkfold_conclusion']))
print('Final dataset version comparison:')
display(artifacts['dataset_version_comparison'])
print('Saved Phase 03 outputs under:')
display(Markdown(f"`{artifacts['phase_dir']}`"))

Leakage check conclusion:


Identifier and label columns are excluded from feature lists. Performance metrics are excluded from the primary without-performance dataset and retained only for auxiliary upper-bound comparison.

GroupKFold readiness conclusion:


Subject-wise GroupKFold is feasible with 35 subjects; recommended splits: 5.

Final dataset version comparison:


,dataset_version,recommended_use,sample_count,identifier_column_count,feature_count,performance_features_included,performance_features_excluded
0,without_performance,Primary Phase 04 ML and Phase 05 HDC dataset,419,6,1176,0,59
1,with_performance,Auxiliary upper-bound and shortcut-learning ri...,419,6,1235,59,0


Saved Phase 03 outputs under:


`E:\hdc-vr-pilot\experiments\phase_03_multimodal_dataset_labeling`